# 8 — CNN


## Imports


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import gc
import random
from time import perf_counter
import numpy as np
import optuna
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score


## Configuration


In [ ]:
from pathlib import Path
# ใช้ project root เดียวกัน ไม่ต้อง import โค้ด pipeline จาก program
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'dataset').is_dir() and (p / 'model').is_dir() and (p / 'program').is_dir())
RAW = ROOT / 'dataset/raw/mit-bih-arrhythmia-database-1.0.0'
DATA = ROOT / 'dataset/processed'
ARTIFACTS = ROOT / 'model/artifacts'
REPORTS = ROOT / 'model/reports'
CLASSES = ['N', 'S', 'V', 'F', 'Q']
SYMBOLS = ['NLRej', 'AaJS', 'VE', 'F', '/fQ']
LABEL_MAP = {s: i for i, group in enumerate(SYMBOLS) for s in group}
SEED = 42
PREPROCESSING = {'version': 1, 'fs': 360, 'left': 90, 'right': 162,
                 'band': [0.5, 40], 'order': 4, 'leads': ['MLII', 'V5']}
MODEL = 'cnn'
TRIALS = 10
BASELINE_PARAMETERS = {'channels': [32, 64, 128], 'kernel': 7, 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.0001}

## Artifact helpers


In [ ]:
def read_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'ไม่พบ {path}; กรุณารัน notebook ขั้นก่อนหน้าก่อน')
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def new_stage(marker):
    # ห้ามเขียนทับผลเดิมเงียบ ๆ เพราะอาจทำให้โมเดลอ้างอิงคนละ split
    if Path(marker).exists():
        raise FileExistsError(f'มีผลขั้นนี้แล้ว: {marker}; โหลดผลเดิมแทนการรันซ้ำ')
    if (ARTIFACTS / 'selection.json').exists():
        raise RuntimeError('ล็อกโมเดลแล้ว ไม่อนุญาตให้เตรียมข้อมูลหรือ tuning ใหม่ใน experiment เดิม')

## Load partition helper


In [ ]:
def load_partition(split, *, allow_test=False):
    if split not in ['train', 'validation', 'test']:
        raise ValueError('Unknown partition')
    if split == 'test' and not allow_test:
        raise PermissionError('Test เปิดได้เฉพาะ final evaluation หลังล็อกโมเดล')
    manifest = read_json(DATA / 'prepared.json')
    folder = DATA / split
    for name in ['metadata.parquet', 'waveforms.npy', 'features.npy']:
        if sha256(folder / name) != manifest['files'][f'{split}/{name}']:
            raise ValueError(f'artifact เปลี่ยน: {split}/{name}')
    meta = pd.read_parquet(folder / 'metadata.parquet')
    waves, x = np.load(folder / 'waveforms.npy'), np.load(folder / 'features.npy')
    if len(meta) != len(waves) or len(meta) != len(x) or meta.beat_id.duplicated().any():
        raise ValueError('beat alignment ไม่ถูกต้อง')
    if not meta.split.eq(split).all() or set(meta.class_id) != set(range(5)):
        raise ValueError('partition หรือ class coverage ไม่ถูกต้อง')
    if not np.array_equal(meta.waveform_index.to_numpy(), np.arange(len(meta))):
        raise ValueError('waveform index ไม่ตรงกับ metadata')
    if waves.shape != (len(meta), 252) or not np.isfinite(waves).all() or not np.isfinite(x).all():
        raise ValueError('ขนาด waveform หรือค่าข้อมูลหลัง preprocessing ไม่ถูกต้อง')
    return {'meta': meta, 'wave': waves, 'x': x, 'y': meta.class_id.to_numpy()}

## Metrics and training weights


In [ ]:
def macro_f1(y, predicted):
    return float(f1_score(y, predicted, labels=list(range(5)), average='macro', zero_division=0))

def weights_from_train(y):
    counts = np.bincount(y, minlength=5)
    if (counts == 0).any():
        raise ValueError('Train ต้องมีครบ 5 คลาส')
    return len(y) / (5 * counts)

## Reproducibility


In [ ]:
def environment():
    packages = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'wfdb', 'xgboost',
                'torch', 'optuna', 'streamlit', 'pyarrow']
    versions = {}
    for package in packages:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = 'not installed'
    return {'python': platform.python_version(), 'platform': platform.platform(),
            'processor': platform.processor(), 'packages': versions}

def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

## Model architecture


In [ ]:
class ECGNetwork(nn.Module):
    def __init__(self, kind, channels, kernel=7, hidden=64, dropout=.3):
        super().__init__()
        layers, previous = [], 1
        for width in channels:
            layers += [nn.Conv1d(previous, width, kernel, padding=kernel // 2),
                       nn.BatchNorm1d(width), nn.ReLU(), nn.MaxPool1d(2)]
            previous = width
        self.conv = nn.Sequential(*layers)
        self.kind = kind
        if kind == 'cnn_bilstm':
            self.lstm = nn.LSTM(previous, hidden, batch_first=True, bidirectional=True)
            previous = hidden * 2
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(previous, 5))

    def forward(self, x):
        x = self.conv(x)
        if self.kind == 'cnn_bilstm':
            _, (hidden, _) = self.lstm(x.transpose(1, 2))
            x = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            x = x.mean(dim=2)
        return self.head(x)

def network(kind, parameters):
    return ECGNetwork(kind, parameters['channels'], parameters.get('kernel', 7),
                      parameters.get('hidden', 64), parameters.get('dropout', .3))

## Prediction


In [ ]:
def probabilities(model, kind, values, device='cpu', batch_size=128):
    outputs = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(values), batch_size):
            batch = torch.as_tensor(values[start:start + batch_size, None, :], device=device)
            outputs.append(model(batch).softmax(dim=1).cpu().numpy())
    return np.concatenate(outputs)

def synchronize(device):
    if str(device).startswith('cuda'):
        torch.cuda.synchronize()

## Training loop and early stopping


In [ ]:
# เก็บ checkpoint เมื่อ Validation Macro F1 ดีขึ้น ไม่ใช้ Test และไม่ refit
def _fit_network(kind, parameters, train, validation, weights, folder, device, batch_size, epochs):
    set_seed()
    model = network(kind, parameters).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=parameters['lr'], weight_decay=parameters['weight_decay'])
    criterion = nn.CrossEntropyLoss(weight=None if weights is None else torch.tensor(weights, dtype=torch.float32, device=device))
    ds = TensorDataset(torch.from_numpy(train['wave'][:, None, :]), torch.tensor(train['y'], dtype=torch.long))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0,
                        generator=torch.Generator().manual_seed(SEED))
    best, stale, best_epoch = -1., 0, 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.
        for values, labels in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(values.to(device)), labels.to(device))
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(labels)
        prediction = probabilities(model, kind, validation['wave'], device, batch_size).argmax(1)
        score = macro_f1(validation['y'], prediction)
        history.append({'epoch': epoch, 'train_loss': loss_sum / len(ds), 'validation_macro_f1': score})
        if score > best:
            best, stale, best_epoch = score, 0, epoch
            # เก็บ epoch ที่ดีที่สุด ไม่ใช้ epoch สุดท้ายโดยอัตโนมัติ
            torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, folder / 'model.pt')
        else:
            stale += 1
        if stale >= 10:
            break
    model.load_state_dict(torch.load(folder / 'model.pt', map_location=device, weights_only=True))
    pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
    return model, best_epoch

## Fit and save one candidate


In [ ]:
# Train เท่านั้นที่ปรับน้ำหนักโมเดล; Validation ใช้ให้คะแนนและเลือก checkpoint
def fit_candidate(kind, weighted, parameters, folder, train, validation, *, epochs=60, rounds=1000):
    folder.mkdir(parents=True, exist_ok=False)
    set_seed()
    is_dl = True
    device = 'cuda' if is_dl and torch.cuda.is_available() else 'cpu'
    if is_dl and device != 'cuda':
        raise RuntimeError('แผนนี้ใช้ GPU สำหรับ DL: ติดตั้ง PyTorch CUDA ก่อนรัน')
    weights = weights_from_train(train['y']) if weighted else None
    started = perf_counter()
    history, best_round, used_batch, oom_batches = ({}, None, None, [])
    for batch_size in [128, 64, 32]:
        try:
            model, best_round = _fit_network(kind, parameters, train, validation, weights, folder, device, batch_size, epochs)
            used_batch = batch_size
            break
        except torch.cuda.OutOfMemoryError:
            oom_batches.append(batch_size)
            gc.collect()
            torch.cuda.empty_cache()
            if batch_size == 32:
                raise
    synchronize(device)
    train_seconds = perf_counter() - started
    values = validation['wave'] if is_dl else validation['x']
    proba = probabilities(model, kind, values, device)
    score = macro_f1(validation['y'], proba.argmax(1))
    result = validation['meta'].copy()
    result['predicted'] = proba.argmax(1)
    for i in range(5):
        result[f'score_{i}'] = proba[:, i]
    result.to_parquet(folder / 'validation_predictions.parquet', index=False)
    info = {'kind': kind, 'weighted': weighted, 'seed': SEED, 'parameters': parameters, 'validation_macro_f1': score, 'train_seconds': train_seconds, 'best_epoch_or_round': best_round, 'batch_size': used_batch, 'oom_batch_attempts': oom_batches, 'device': device, 'gpu': torch.cuda.get_device_name(0) if device == 'cuda' else None, 'class_weights': None if weights is None else weights.tolist(), 'prepared_hash': sha256(DATA / 'prepared.json'), 'environment': environment()}
    write_json(folder / 'info.json', info)
    del model
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()
    return info

## Hyperparameter search space


In [ ]:
def search_parameters(trial, kind):
    width = trial.suggest_categorical('width', ['small', 'large'])
    channels = [16, 32, 64] if width == 'small' else [32, 64, 128]
    values = {'channels': channels, 'kernel': 7, 'dropout': trial.suggest_float('dropout', 0.1, 0.5), 'lr': trial.suggest_float('lr', 0.0001, 0.003, log=True), 'weight_decay': trial.suggest_float('weight_decay', 1e-06, 0.001, log=True)}
    values['kernel'] = trial.suggest_categorical('kernel', [5, 7, 11])
    return values

## Load Train and Validation


In [ ]:
# ไม่อ่าน Test และไม่แบ่ง inner validation
new_stage(ARTIFACTS / 'selection.json')
train = load_partition('train')
validation = load_partition('validation')

## Baseline


In [ ]:
# ฝึก baseline ก่อน เพื่อเทียบกับผล tuning โดยใช้ split เดียวกัน
runs = {}
for weighted in [False, True]:
    name = f'{MODEL}_{"weighted" if weighted else "unweighted"}'
    folder = ARTIFACTS / name
    new_stage(folder / 'selected.json')
    if folder.exists():
        raise FileExistsError(f'{folder} มีผลอยู่แล้ว ตรวจหรือเก็บสำรองก่อนเริ่มใหม่')
    folder.mkdir(parents=True)
    baseline = fit_candidate(MODEL, weighted, BASELINE_PARAMETERS, folder / 'baseline', train, validation)
    runs[name] = {'folder': folder, 'weighted': weighted, 'baseline': baseline,
                  'candidates': [('baseline', baseline)], 'tuning_seconds': 0.}

## Tuning on Validation


In [ ]:
# แต่ละ trial ฝึกด้วย hyperparameters ของ trial โดยใช้ seed 42 เดียวกัน
# หลัง tuning เลือก checkpoint ที่มีแล้ว ไม่ฝึกใหม่อีกครั้ง
for name, run in runs.items():
    folder = run['folder']
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED),
                               storage=f'sqlite:///{(folder / "study.sqlite3").as_posix()}', study_name=name)
    def objective(trial):
        run_id = f'trial_{trial.number:02d}'
        params = search_parameters(trial, MODEL)
        try:
            info = fit_candidate(MODEL, run['weighted'], params, folder / run_id, train, validation)
        except Exception as error:
            trial.set_user_attr('failure', f'{type(error).__name__}: {error}')
            write_json(folder / run_id / 'failure.json', {'error': str(error), 'parameters': params})
            raise
        run['candidates'].append((run_id, info))
        return info['validation_macro_f1']
    started = perf_counter()
    try:
        study.optimize(objective, n_trials=TRIALS, catch=(RuntimeError, ValueError), n_jobs=1)
    finally:
        run['tuning_seconds'] = perf_counter() - started
        study.trials_dataframe().to_csv(folder / 'trials.csv', index=False)

## Select and save candidates


In [ ]:
# เลือกตัวแทนแต่ละ configuration จาก Validation เท่านั้น
selected_candidates = []
for name, run in runs.items():
    selected_id, info = max(run['candidates'], key=lambda pair: pair[1]['validation_macro_f1'])
    selected = {'configuration': name, 'candidate': selected_id,
                'validation_macro_f1': info['validation_macro_f1'],
                'baseline_macro_f1': run['baseline']['validation_macro_f1'],
                'tuning_seconds': run['tuning_seconds'], 'prepared_hash': info['prepared_hash']}
    write_json(run['folder'] / 'selected.json', selected)
    selected_candidates.append(selected)
pd.DataFrame(selected_candidates)